## Импорты инсталлы

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
import warnings
from typing import TypedDict, Literal, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import json
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Any
import subprocess

pd.set_option('display.max_columns', 100)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
load_dotenv()

True

In [3]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API Key: {API_KEY is not None}")

API Key: True


## Описание

| Название колонки              | Описание                                                             |
| ----------------------------- | -------------------------------------------------------------------- |
| `id`                          | Уникальный идентификатор объявления.                                 |
| `name`                        | Название объявления.                                                 |
| `description`                 | Текстовое описание объявления.                                       |
| `property_type`               | Тип недвижимости, например apartment, house, private room.           |
| `room_type`                   | Тип сдаваемого помещения, например Entire home/apt или Private room. |
| `accommodates`                | Сколько гостей может разместить объект.                              |
| `bathrooms`                   | Информация о ванных комнатах.                                        |
| `bedrooms`                    | Информация о спальнях.                                               |
| `beds`                        | Количество кроватей.                                                 |
| `amenities`                   | Список удобств, предоставляемых в объекте.                           |
| `host_id`                     | Уникальный идентификатор хозяина.                                    |
| `host_name`                   | Имя хозяина.                                                         |
| `host_since`                  | Дата регистрации хозяина на Airbnb.                                  |
| `host_location`               | Местоположение хозяина.                                              |
| `host_about`                  | Информация хозяина о себе.                                           |
| `host_response_time`          | Время ответа хозяина.                                                |
| `host_response_rate`          | Доля ответов хозяина.                                                |
| `host_acceptance_rate`        | Доля подтвержденных хозяином бронирований.                           |
| `host_is_superhost`           | Является ли хозяин супер-хозяином.                                   |
| `host_neighbourhood`          | Район проживания хозяина, если указан.                               |
| `host_listings_count`         | Количество объявлений у хозяина.                                     |
| `host_total_listings_count`   | Общее число объявлений хозяина.                                      |
| `host_verifications`          | Способы верификации хозяина.                                         |
| `host_has_profile_pic`        | Есть ли у хозяина фото профиля.                                      |
| `host_identity_verified`      | Подтверждена ли личность хозяина.                                    |
| `latitude`                    | Географическая широта объекта.                                       |
| `longitude`                   | Географическая долгота объекта.                                      |
| `neighbourhood_overview`      | Описание района, где находится объект.                               |
| `city`                        | Город, где расположен объект.                                        |
| `price`                       | Цена за ночь.                                                        |
| `has_availability`            | Есть ли у объекта доступные даты.                                    |
| `availability_30`             | Количество доступных дней в ближайшие 30 дней.                       |
| `availability_60`             | Количество доступных дней в ближайшие 60 дней.                       |
| `availability_90`             | Количество доступных дней в ближайшие 90 дней.                       |
| `availability_365`            | Количество доступных дней в ближайшие 365 дней.                      |
| `availability_eoy`            | Доступность объекта на конец года.                                   |
| `number_of_reviews`           | Общее количество отзывов.                                            |
| `number_of_reviews_ltm`       | Количество отзывов за последние 12 месяцев.                          |
| `number_of_reviews_l30d`      | Количество отзывов за последние 30 дней.                             |
| `number_of_reviews_ly`        | Количество отзывов за последний год.                                 |
| `first_review`                | Дата первого отзыва.                                                 |
| `last_review`                 | Дата последнего отзыва.                                              |
| `review_scores_rating`        | Общий рейтинг по отзывам.                                            |
| `review_scores_accuracy`      | Оценка точности описания объекта.                                    |
| `review_scores_cleanliness`   | Оценка чистоты.                                                      |
| `review_scores_checkin`       | Оценка удобства заселения.                                           |
| `review_scores_communication` | Оценка качества коммуникации с хозяином.                             |
| `review_scores_location`      | Оценка расположения.                                                 |
| `review_scores_value`         | Оценка соотношения цены и качества.                                  |
| `reviews_per_month`           | Среднее количество отзывов в месяц.                                  |
| `estimated_occupancy_l365d`   | Оценочная заполняемость за последние 365 дней.                       |
| `estimated_revenue_l365d`     | Оценочная выручка за последние 365 дней.                             |


## Обзор

In [4]:
data = pd.read_csv("rent_predictions/airbnb_train.csv")
data.head()

,id,name,description,neighborhood_overview,host_id,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month,city
0,246350,Private Room w/ Patio Downtown Toronto,This eclectic downtown apartment is comfortabl...,Corktown Toronto.,129487451,Ashley,2017-05-08,"Toronto, Canada","Hi I’m Ashley, I’m a mum of 2 little ones who ...",within an hour,100%,100%,t,NaN,29.0,33.0,"['email', 'phone']",t,t,43.655580,-79.364260,Private room in rental unit,Private room,1,1.0,1.0,1.0,"[""Hair dryer"", ""Room-darkening shades"", ""Keypa...",43.0,t,0,29,59,149,9,2,0,149.0,2.0,112.0,4816.0,2021-07-01,2024-12-20,4.67,4.89,4.89,4.89,4.89,4.78,4.56,0.20,Toronto
1,11616,Habitación 5 -3,Enjoy the simplicity of this quiet and central...,NaN,18191427,David,2014-07-16,"Barcelona, Spain",NaN,within an hour,99%,97%,f,NaN,27.0,28.0,"['email', 'phone']",t,t,41.391680,2.161700,Private room in rental unit,Private room,2,2.0,1.0,1.0,"[""Shampoo"", ""Conditioner"", ""Other electric sto...",60.0,t,26,50,75,347,5,5,3,282.0,0.0,30.0,1800.0,2025-01-01,2025-02-19,4.00,4.60,4.00,4.00,4.40,4.60,4.00,2.27,Barcelona
2,70489,Royal Garden Waikiki Studio with Kitchinette -20%,Our condo is in the Club Wyndham building and ...,NaN,11916207,Alan,2014-02-03,"Campbell, CA","English guy married to a Philipinno lady, live...",within an hour,100%,100%,f,NaN,2.0,2.0,"['email', 'phone']",t,t,21.284510,-157.830078,Entire condo,Entire home/apt,2,1.0,0.0,1.0,"[""Freezer"", ""Hair dryer"", ""Bathtub"", ""Shared s...",141.0,t,20,35,57,194,9,0,0,168.0,2.0,0.0,0.0,2021-11-22,2024-02-17,4.78,5.00,4.89,5.00,4.89,5.00,5.00,0.22,Hawaii
3,89610,Spacious 3 bedroom Apartment,A spacious and beautiful 3 bedroom & 2 bathroo...,NaN,36818617,My London,2015-06-26,"London, United Kingdom",NaN,within an hour,100%,100%,f,Paddington,70.0,111.0,"['email', 'phone', 'work_email']",t,t,51.546420,-0.220100,Entire serviced apartment,Entire home/apt,7,2.0,3.0,4.0,"[""Cooking basics"", ""Fire extinguisher"", ""Dryer...",204.0,t,9,39,69,69,1,1,0,NaN,NaN,NaN,NaN,2024-03-27,2024-03-27,4.00,4.00,5.00,4.00,5.00,4.00,4.00,0.11,London
4,43669,Villa zafira with private pool,Leave behind any worries with this spacious an...,NaN,364602280,Panagiotis,2020-08-27,NaN,NaN,NaN,NaN,NaN,f,NaN,1.0,2.0,"['email', 'phone']",t,t,35.381583,24.611163,Entire villa,Entire home/apt,6,3.0,3.0,3.0,"[""Free parking on premises"", ""Dishes and silve...",299.0,t,30,60,90,337,1,1,0,NaN,NaN,NaN,NaN,2024-07-27,2024-07-27,5.00,5.00,5.00,5.00,5.00,4.00,5.00,0.19,Crete


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 208856 entries, 0 to 208855
Data columns (total 52 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           208856 non-null  int64  
 1   name                         208856 non-null  str    
 2   description                  203983 non-null  str    
 3   neighborhood_overview        101816 non-null  str    
 4   host_id                      208856 non-null  int64  
 5   host_name                    208764 non-null  str    
 6   host_since                   208757 non-null  str    
 7   host_location                155577 non-null  str    
 8   host_about                   108194 non-null  str    
 9   host_response_time           176895 non-null  str    
 10  host_response_rate           176895 non-null  str    
 11  host_acceptance_rate         190886 non-null  str    
 12  host_is_superhost            203015 non-null  str    
 13  host_neigh

In [6]:
data.duplicated().sum()

np.int64(0)

In [7]:
data.isna().sum().sort_values(ascending=False).loc[lambda x: x > 0]

host_neighbourhood             126725
availability_eoy               111467
number_of_reviews_ly           111467
estimated_occupancy_l365d      111467
estimated_revenue_l365d        111467
neighborhood_overview          107040
host_about                     100662
host_location                   53279
review_scores_checkin           40659
review_scores_value             40658
review_scores_location          40657
review_scores_communication     40654
review_scores_cleanliness       40653
review_scores_accuracy          40653
review_scores_rating            40646
first_review                    40645
last_review                     40645
reviews_per_month               40645
host_response_rate              31961
host_response_time              31961
host_acceptance_rate            17970
host_is_superhost                5841
description                      4873
has_availability                 1833
beds                              396
bedrooms                          275
host_identit

Как мы видим в датасете есть пропуски во многих колонках, нет дубликатов, многие признаки, например `amenities` или любое `description` требует предварительной обработки, чтобы их можно было использовать 

Также можно заметить, что некоторые признаки нельзя использовать т.к они являются признаками полученнным постфактум

## Оркестратор

Зафиксируем output каждого агента

In [8]:
# {
#   "agent": "agent_name",
#   "status": "success",
#   "skipped": false,
#   "summary": "short summary",
#   "decisions": {},
#   "artifacts": {},
#   "next_input": {},
#   "reason": null
# }

Зафиксируем базовый state

<!-- {
  "meta": {
    "run_id": null,
    "session_id": null,
    "created_at": null,
    "status": "initialized",
    "current_step": null,
    "completed_steps": []
  },
  "dataset": {
    "dataset_path": null,
    "dataset_name": null,
    "target_column": null,
    "task_type": null,
    "row_count": null,
    "column_count": null,
    "feature_columns": [],
    "text_columns": [],
    "categorical_columns": [],
    "numerical_columns": []
  },
  "routing": {
    "next_agent": null,
    "skip_text_branch": false,
    "retry_modeling": false,
    "retry_count": 0,
    "max_retries": 1,
    "feature_selection_needed": false,
    "report_ready": false,
    "stop_reason": null
  },
  "agent_outputs": {
    "data_description": null,
    "tabular_preparation": null,
    "text_preparation": null,
    "feature_merge": null,
    "modeling": null,
    "feature_selection": null,
    "reporting": null
  },
  "artifacts": {
    "clean_dataset_path": null,
    "tabular_features_path": null,
    "text_features_path": null,
    "merged_features_path": null,
    "model_path": null,
    "metrics_path": null,
    "report_path": null
  },
  "history": [],
  "errors": [],
  "best_result": {
    "best_model_name": null,
    "best_score": null,
    "best_metric_name": null,
    "comparison_table_path": null
  },
  "persistent_memory": {
    "previous_best_model_path": null,
    "previous_best_score": null,
    "previous_run_id": null
  }
} -->

## Оркестратор

In [9]:
from typing import TypedDict, Literal, Any, Annotated
import operator


class AgentState(TypedDict, total=False):
    run_id: str | None

    dataset_path: str | None
    target_column: str | None
    task_type: Literal["classification", "regression", "auto"]

    schema: dict[str, list[str]]
    data_summary: dict[str, Any]
    prep_summary: dict[str, Any]
    text_summary: dict[str, Any]
    modeling_summary: dict[str, Any]

    artifacts: dict[str, str | None]
    logs: Annotated[list[dict[str, Any]], operator.add]

    current_step: str | None
    next_action: str | None
    orchestration_decision: dict[str, Any]
    orchestration_history: Annotated[list[dict[str, Any]], operator.add]

    previous_run: dict[str, Any]
    previous_best_model: dict[str, Any]

    retry_count: int
    max_retries: int
    status: str
    error: str | None

In [10]:
model = "deepseek/deepseek-v4-flash"
model = 'google/gemma-4-26b-a4b-it'

llm = ChatOpenAI(
    model=model,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=700,
)

In [11]:
response = llm.invoke("Hello!")
print(response.content)

Hello! How can I help you today?


## Делаем поиск датасета и предыдущего рана

In [12]:
class BashInput(BaseModel):
    command: str = Field(
        description="Bash command to execute."
    )

@tool(args_schema=BashInput)
def bash_tool(command: str) -> dict[str, Any]:
    """
    Executes a bash command and returns stdout, stderr and return code.
    Use it for filesystem operations:
    - finding csv files
    - listing artifact folders
    - reading json files
    """

    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )

        return {
            "status": "success" if result.returncode == 0 else "failed",
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
            "returncode": result.returncode,
        }

    except Exception as e:
        return {
            "status": "failed",
            "stdout": "",
            "stderr": str(e),
            "returncode": -1,
        }

In [13]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are an LLM-based ML Workflow Orchestrator Agent.

Your role:
- analyze the current AgentState
- decide what should happen next
- choose which specialized agent should be called
- explain the reason for your decision
- handle errors, retries, fallback logic, and finishing conditions
- use available tools when filesystem inspection is needed
- find the input dataset path when it is missing
- inspect artifact folders to recover previous run information

You do NOT train models directly.
You do NOT preprocess data directly.
You do NOT create reports directly.
You only orchestrate the workflow.

Available tools:
- bash_tool: use it only for safe filesystem inspection, such as finding CSV files, listing artifact folders, or reading JSON metadata.

Available decisions:
- run_data_description
- run_tabular_preparation
- run_text_preparation
- merge_features
- run_modeling
- run_improvement
- run_reporting
- retry_current_step
- use_fallback
- finish
- fail

Specialized agents:
- data_description_agent
- tabular_preparation_agent
- text_preparation_agent
- merge_features
- modeling_agent
- improvement_agent
- reporting_agent

A step is completed ONLY by the following exact rules:

1. data_description_agent is completed if:
   - data_summary contains key "n_rows"
   - data_summary contains key "n_columns"
   - schema contains key "numeric"
   - schema contains key "categorical"
   - schema contains key "text"
   - schema contains key "datetime"

2. tabular_preparation_agent is completed if:
   - prep_summary contains key "missing_strategy"
   - prep_summary contains key "encoding"
   - prep_summary contains key "scaling"
   - prep_summary contains key "created_features"

3. text_preparation_agent is completed if:
   - schema["text"] is an empty list
   OR
   - text_summary["used"] is true

4. merge_features is completed if:
   - artifacts contains key "prepared_dataset_path"
   - artifacts["prepared_dataset_path"] is not null

5. modeling_agent is completed if:
   - modeling_summary contains key "best_model_name"
   - modeling_summary["best_model_name"] is not null

6. reporting_agent is completed if:
   - artifacts contains key "final_report_path"
   - artifacts["final_report_path"] is not null

Decision order:
1. If error is not null and retry_count < max_retries, choose retry_current_step.
2. If error is not null and retry_count >= max_retries, choose use_fallback or fail.
3. If data_description_agent is not completed, choose run_data_description.
4. Else if tabular_preparation_agent is not completed, choose run_tabular_preparation.
5. Else if text_preparation_agent is not completed, choose run_text_preparation.
6. Else if merge_features is not completed, choose merge_features.
7. Else if modeling_agent is not completed, choose run_modeling.
8. Else if modeling_summary["retry_recommended"] is true and retry_count < max_retries, choose run_improvement.
9. Else if reporting_agent is not completed, choose run_reporting.
10. Else choose finish.

Strict rules:
- Do not invent additional completion requirements.
- Do not say a step is "not fully populated" if the exact completion keys exist.
- Do not choose run_data_description if data_summary has n_rows and n_columns and schema has numeric, categorical, text, datetime.
- Do not choose run_tabular_preparation if prep_summary has missing_strategy, encoding, scaling, created_features.
- Do not choose run_modeling before merge_features is completed.
- tabular_dataset_path is not the same as prepared_dataset_path.
- Modeling can start only after prepared_dataset_path exists.
- Do not choose retry_current_step if error is null.
- Do not choose use_fallback if error is null.
- Use bash_tool only for safe filesystem inspection.
- Do not delete, move, overwrite, or modify files using bash_tool.
- For dataset search, prefer CSV files inside ./data.
- Ignore CSV files inside ./artifacts.
- Ignore predictions.csv, metrics.csv, and temporary output CSV files.
- Return only valid JSON when a structured decision is requested.
"""

In [14]:
from langchain.agents import create_agent

tools = [bash_tool]

orchestrator_tool_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
)

In [15]:
import re


def extract_csv_path(text: str) -> str | None:
    match = re.search(r"([./\w\-/]+\.csv)", text)
    if match:
        return match.group(1)
    return None


def dataset_finder_node(state: AgentState) -> AgentState:
    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the main CSV dataset in the current project folder. "
                    "Use bash_tool if needed. "
                    "Prefer files inside ./data. "
                    "Ignore files inside ./artifacts. "
                    "Ignore predictions.csv, metrics.csv, and temporary output files. "
                    "Return only the dataset path as plain text. "
                    "Do not add explanations."
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()

    dataset_path = extract_csv_path(agent_response)

    # fallback через bash, если агент ответил криво
    fallback_used = False
    fallback_result = None

    if not dataset_path:
        fallback_used = True
        fallback_result = bash_tool.invoke({
            "command": (
                'find . -type f -name "*.csv" '
                '! -path "./artifacts/*" '
                '! -name "predictions.csv" '
                '! -name "metrics.csv" '
                '2>/dev/null | head -n 1'
            )
        })

        dataset_path = fallback_result.get("stdout", "").strip()
        if dataset_path:
            dataset_path = dataset_path.splitlines()[0].strip()

    if not dataset_path:
        return {
            "error": f"Dataset path was not found. Agent returned: {agent_response}",
            "logs": [{
                "agent": "dataset_finder",
                "status": "failed",
                "skipped": False,
                "summary": "Dataset finder failed to locate a valid CSV file.",
                "decisions": {
                    "agent_response": agent_response,
                    "fallback_used": fallback_used,
                    "fallback_result": fallback_result,
                },
                "artifacts": {},
                "next_input": {},
                "reason": "No valid CSV path found.",
            }],
        }

    return {
        "dataset_path": dataset_path,
        "logs": [{
            "agent": "dataset_finder",
            "status": "success",
            "skipped": False,
            "summary": f"Dataset found: {dataset_path}",
            "decisions": {
                "agent_response": agent_response,
                "fallback_used": fallback_used,
                "fallback_result": fallback_result,
            },
            "artifacts": {},
            "next_input": {
                "dataset_path": dataset_path,
            },
            "reason": None,
        }],
    }

In [17]:
initial_state = {
    "run_id": None,
    "dataset_path": None,
    "target_column": "target",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": [],
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

In [18]:
dataset_result = dataset_finder_node(initial_state)

print("dataset_path:", dataset_result.get("dataset_path"))
print("error:", dataset_result.get("error"))
print("log:", dataset_result["logs"][-1])

dataset_path: ./rent_predictions/airbnb_train.csv
error: None
log: {'agent': 'dataset_finder', 'status': 'success', 'skipped': False, 'summary': 'Dataset found: ./rent_predictions/airbnb_train.csv', 'decisions': {'agent_response': 'thought\n./rent_predictions/airbnb_train.csv', 'fallback_used': False, 'fallback_result': None}, 'artifacts': {}, 'next_input': {'dataset_path': './rent_predictions/airbnb_train.csv'}, 'reason': None}


In [19]:
def extract_json(text: str) -> dict:
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    return json.loads(text)

In [20]:
import json
import re


def extract_json_safe(text: str) -> dict | None:
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None


def previous_run_finder_node(state: AgentState) -> AgentState:
    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the latest previous ML run inside the artifacts folder. "
                    "Use bash_tool if needed. "
                    "Look for run directories that may contain metrics.json and best_model.pkl. "
                    "Return only valid JSON, no markdown, no explanations. "
                    "JSON format:\n"
                    "{\n"
                    '  "latest_run_path": "path or null",\n'
                    '  "metrics_path": "path or null",\n'
                    '  "best_model_path": "path or null"\n'
                    "}"
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()
    parsed = extract_json_safe(agent_response)

    # Fallback: если LLM не вернула нормальный JSON,
    # сами через bash ищем последний run.
    fallback_used = False
    fallback_result = None

    if parsed is None:
        fallback_used = True

        fallback_result = bash_tool.invoke({
            "command": (
                'latest_run=$(ls -td artifacts/runs/* 2>/dev/null | head -n 1); '
                'if [ -z "$latest_run" ]; then '
                'echo \'{"latest_run_path": null, "metrics_path": null, "best_model_path": null}\'; '
                'else '
                'metrics_path="$latest_run/metrics.json"; '
                'model_path="$latest_run/best_model.pkl"; '
                'if [ ! -f "$metrics_path" ]; then metrics_path=null; fi; '
                'if [ ! -f "$model_path" ]; then model_path=null; fi; '
                'echo "{\\"latest_run_path\\": \\"$latest_run\\", \\"metrics_path\\": \\"$metrics_path\\", \\"best_model_path\\": \\"$model_path\\"}"; '
                'fi'
            )
        })

        parsed = extract_json_safe(fallback_result.get("stdout", ""))

    # Если даже fallback не сработал — считаем, что прошлого run нет.
    if parsed is None:
        parsed = {
            "latest_run_path": None,
            "metrics_path": None,
            "best_model_path": None,
        }

    latest_run_path = parsed.get("latest_run_path")
    metrics_path = parsed.get("metrics_path")
    best_model_path = parsed.get("best_model_path")

    if latest_run_path in [None, "null", "None", ""]:
        previous_run = {
            "exists": False,
            "path": None,
        }

        previous_best_model = {}

        log_record = {
            "agent": "previous_run_finder",
            "status": "success",
            "skipped": True,
            "summary": "No previous run was found.",
            "decisions": {
                "agent_response": agent_response,
                "parsed": parsed,
                "fallback_used": fallback_used,
                "fallback_result": fallback_result,
            },
            "artifacts": {},
            "next_input": {
                "previous_run": previous_run,
                "previous_best_model": previous_best_model,
            },
            "reason": "No previous run directory exists.",
        }

        return {
            "previous_run": previous_run,
            "previous_best_model": previous_best_model,
            "logs": [log_record],
        }

    previous_metrics = {}

    if metrics_path not in [None, "null", "None", ""]:
        metrics_result = bash_tool.invoke({
            "command": f'cat "{metrics_path}" 2>/dev/null'
        })

        metrics_raw = metrics_result.get("stdout", "").strip()

        if metrics_raw:
            try:
                previous_metrics = json.loads(metrics_raw)
            except json.JSONDecodeError:
                previous_metrics = {
                    "raw_metrics": metrics_raw,
                }

    previous_run = {
        "exists": True,
        "path": latest_run_path,
        "metrics_path": metrics_path,
        "best_model_path": best_model_path,
    }

    previous_best_model = {
        "run_path": latest_run_path,
        "model_path": best_model_path,
        "metrics_path": metrics_path,
        "metrics": previous_metrics,
    }

    log_record = {
        "agent": "previous_run_finder",
        "status": "success",
        "skipped": False,
        "summary": f"Previous run found: {latest_run_path}",
        "decisions": {
            "agent_response": agent_response,
            "parsed": parsed,
            "fallback_used": fallback_used,
            "fallback_result": fallback_result,
            "previous_metrics": previous_metrics,
        },
        "artifacts": {},
        "next_input": {
            "previous_run": previous_run,
            "previous_best_model": previous_best_model,
        },
        "reason": None,
    }

    return {
        "previous_run": previous_run,
        "previous_best_model": previous_best_model,
        "logs": [log_record],
    }

In [21]:
previous_result = previous_run_finder_node(initial_state)

print("previous_run:", previous_result.get("previous_run"))
print("previous_best_model:", previous_result.get("previous_best_model"))
print("error:", previous_result.get("error"))
print("log:", previous_result["logs"][-1])

previous_run: {'exists': False, 'path': None}
previous_best_model: {}
error: None
log: {'agent': 'previous_run_finder', 'status': 'success', 'skipped': True, 'summary': 'No previous run was found.', 'decisions': {'agent_response': '{\n  "latest_run_path": null,\n  "metrics_path": null,\n  "best_model_path": null\n}', 'parsed': {'latest_run_path': None, 'metrics_path': None, 'best_model_path': None}, 'fallback_used': False, 'fallback_result': None}, 'artifacts': {}, 'next_input': {'previous_run': {'exists': False, 'path': None}, 'previous_best_model': {}}, 'reason': 'No previous run directory exists.'}


In [22]:
def join_init_node(state: AgentState) -> AgentState:
    log_record = {
        "agent": "join_init",
        "status": "success",
        "skipped": False,
        "summary": "Initialization steps completed: dataset search and previous run search are done.",
        "decisions": {
            "dataset_path": state.get("dataset_path"),
            "previous_run": state.get("previous_run", {}),
            "previous_best_model": state.get("previous_best_model", {}),
        },
        "artifacts": {},
        "next_input": {},
        "reason": None,
    }

    return {
        "current_step": "join_init",
        "logs": [log_record],
    }

## 

In [23]:
from typing import Literal, Optional, Any
from pydantic import BaseModel, Field


class OrchestratorDecision(BaseModel):
    decision: Literal[
        "run_data_description",
        "run_tabular_preparation",
        "run_text_preparation",
        "merge_features",
        "run_modeling",
        "run_improvement",
        "run_reporting",
        "retry_current_step",
        "use_fallback",
        "finish",
        "fail",
    ] = Field(
        description="Следующее действие, которое выбрал orchestrator."
    )

    reason: str = Field(
        description="Краткое объяснение, почему выбрано именно это действие."
    )

    selected_agent: Optional[str] = Field(
        default=None,
        description="Название агента, которого нужно вызвать следующим."
    )

    input_needed: dict[str, Any] = Field(
        default_factory=dict,
        description="Какие входные данные нужны следующему агенту."
    )

    expected_output: list[str] = Field(
        default_factory=list,
        description="Какие результаты ожидаются от следующего агента."
    )

    risk_notes: list[str] = Field(
        default_factory=list,
        description="Риски или важные замечания по текущему шагу."
    )

In [24]:
test_decision = OrchestratorDecision(
    decision="run_data_description",
    reason="Первичный анализ данных еще не выполнен.",
    selected_agent="data_description_agent",
    input_needed={
        "dataset_path": "data/raw.csv",
        "target_column": "target"
    },
    expected_output=[
        "schema",
        "data_summary"
    ],
    risk_notes=[
        "Тип задачи пока может быть неизвестен."
    ]
)

test_decision

OrchestratorDecision(decision='run_data_description', reason='Первичный анализ данных еще не выполнен.', selected_agent='data_description_agent', input_needed={'dataset_path': 'data/raw.csv', 'target_column': 'target'}, expected_output=['schema', 'data_summary'], risk_notes=['Тип задачи пока может быть неизвестен.'])

In [25]:
import json
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import ValidationError


def extract_json(text: str) -> dict:
    """
    Достает JSON из ответа модели.
    Нужно на случай, если модель случайно вернет ```json ... ```.
    """
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    return json.loads(text)


def call_orchestrator_llm(state: dict) -> OrchestratorDecision:
    """
    Sends current AgentState to the LLM orchestrator
    and returns a validated OrchestratorDecision.
    """

    state_for_llm = {
        "run_id": state.get("run_id"),
        "dataset_path": state.get("dataset_path"),
        "target_column": state.get("target_column"),
        "task_type": state.get("task_type"),

        "schema": state.get("schema", {}),
        "data_summary": state.get("data_summary", {}),
        "prep_summary": state.get("prep_summary", {}),
        "text_summary": state.get("text_summary", {}),
        "modeling_summary": state.get("modeling_summary", {}),
        "artifacts": state.get("artifacts", {}),

        "available_keys": {
            "schema_keys": list(state.get("schema", {}).keys()),
            "data_summary_keys": list(state.get("data_summary", {}).keys()),
            "prep_summary_keys": list(state.get("prep_summary", {}).keys()),
            "text_summary_keys": list(state.get("text_summary", {}).keys()),
            "modeling_summary_keys": list(state.get("modeling_summary", {}).keys()),
            "artifact_keys": list(state.get("artifacts", {}).keys()),
        },

        "logs_count": len(state.get("logs", [])),
        "current_step": state.get("current_step"),
        "next_action": state.get("next_action"),
        "orchestration_history": state.get("orchestration_history", [])[-5:],

        "retry_count": state.get("retry_count", 0),
        "max_retries": state.get("max_retries", 1),
        "status": state.get("status"),
        "error": state.get("error"),
    }

    response = llm.invoke(
        [
            SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    "Analyze the current AgentState and choose the next action.\n\n"
                    "Return ONLY valid JSON. Do not use markdown. Do not add explanations outside JSON.\n\n"
                    "JSON format:\n"
                    "{\n"
                    '  "decision": "run_data_description",\n'
                    '  "reason": "short reason",\n'
                    '  "selected_agent": "data_description_agent",\n'
                    '  "input_needed": {},\n'
                    '  "expected_output": [],\n'
                    '  "risk_notes": []\n'
                    "}\n\n"
                    f"AgentState:\n{json.dumps(state_for_llm, ensure_ascii=False, indent=2)}"
                )
            ),
        ]
    )

    raw_text = response.content
    parsed = extract_json(raw_text)

    try:
        return OrchestratorDecision(**parsed)
    except ValidationError as e:
        raise ValueError(
            f"LLM returned invalid OrchestratorDecision.\n"
            f"Validation error: {e}\n"
            f"Raw response: {raw_text}"
        )

In [26]:
state_after_tabular = {
    **initial_state,
    "schema": {
        "numeric": ["feature_1", "feature_2"],
        "categorical": ["category_1"],
        "text": [],
        "datetime": [],
    },
    "data_summary": {
        "n_rows": 1000,
        "n_columns": 4,
    },
    "prep_summary": {
        "duplicates_action": "dropped",
        "missing_strategy": {"numeric": "median", "categorical": "mode"},
        "outlier_strategy": "winsorization",
        "encoding": "one_hot_encoding",
        "scaling": "standard_scaler",
        "target_transform": None,
        "created_features": ["feature_1_x_feature_2", "feature_2_is_missing"],
    },
    "artifacts": {
        "tabular_dataset_path": "artifacts/tabular_prepared.csv",
    },
    "error": None,
}

In [27]:
ORCHESTRATOR_SYSTEM_PROMPT += """

Strict merge rule:
- artifacts.tabular_dataset_path means tabular preparation is completed.
- artifacts.prepared_dataset_path means feature merging is completed.
- These are different artifacts.
- Never choose run_modeling if artifacts.prepared_dataset_path is missing.
- If artifacts.tabular_dataset_path exists but artifacts.prepared_dataset_path is missing, choose merge_features.
"""

In [28]:
decision = call_orchestrator_llm(state_after_tabular)
decision

OrchestratorDecision(decision='merge_features', reason='data_description_agent, tabular_preparation_agent, and text_preparation_agent are all completed based on the state. tabular_dataset_path exists in artifacts, but prepared_dataset_path is missing, so merge_features must be run next.', selected_agent='merge_features', input_needed={'tabular_dataset_path': 'artifacts/tabular_prepared.csv'}, expected_output=['artifacts.prepared_dataset_path'], risk_notes=[])

In [29]:
print("schema:", state_after_tabular.get("schema"))
print("data_summary:", state_after_tabular.get("data_summary"))
print("prep_summary:", state_after_tabular.get("prep_summary"))
print("artifacts:", state_after_tabular.get("artifacts"))
print("error:", state_after_tabular.get("error"))

schema: {'numeric': ['feature_1', 'feature_2'], 'categorical': ['category_1'], 'text': [], 'datetime': []}
data_summary: {'n_rows': 1000, 'n_columns': 4}
prep_summary: {'duplicates_action': 'dropped', 'missing_strategy': {'numeric': 'median', 'categorical': 'mode'}, 'outlier_strategy': 'winsorization', 'encoding': 'one_hot_encoding', 'scaling': 'standard_scaler', 'target_transform': None, 'created_features': ['feature_1_x_feature_2', 'feature_2_is_missing']}
artifacts: {'tabular_dataset_path': 'artifacts/tabular_prepared.csv'}
error: None


In [287]:
print("status:", final_state.get("status"))
print("error:", final_state.get("error"))
print("artifacts:", final_state.get("artifacts"))
print("modeling_summary:", final_state.get("modeling_summary"))

status: success
error: None
artifacts: {'data_profile_path': 'artifacts/data_profile.json', 'tabular_dataset_path': 'artifacts/tabular_prepared.csv', 'prepared_dataset_path': 'artifacts/prepared_dataset.csv', 'best_model_path': 'artifacts/best_model.pkl', 'predictions_path': 'artifacts/predictions.csv', 'metrics_path': 'artifacts/metrics.json', 'final_report_path': 'artifacts/final_report.md'}
modeling_summary: {'models_tested': ['LogisticRegression', 'RandomForestClassifier', 'GradientBoostingClassifier'], 'primary_metric': 'f1', 'best_model_name': 'RandomForestClassifier', 'validation_metrics': {'f1': 0.83, 'roc_auc': 0.88}, 'test_metrics': {'f1': 0.81, 'roc_auc': 0.86}, 'retry_recommended': False, 'improvement_mode': None, 'business_interpretation': 'The model can identify the positive class with acceptable quality.'}


In [289]:
test_state = {
    "run_id": "test-run-1",
    "dataset_path": "data/raw.csv",
    "target_column": "target",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": []
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

In [290]:
decision = call_orchestrator_llm(test_state)
decision

OrchestratorDecision(decision='run_data_description', reason='The data_description_agent is not completed because data_summary and schema are empty.', selected_agent='data_description_agent', input_needed={'dataset_path': 'data/raw.csv', 'target_column': 'target'}, expected_output=['data_summary', 'schema'], risk_notes=[])

In [30]:
def orchestrator_agent_node(state: AgentState) -> AgentState:
    decision = call_orchestrator_llm(state)
    decision_dict = decision.model_dump()

    log_record = {
        "agent": "orchestrator_agent",
        "status": "success",
        "skipped": False,
        "summary": decision.reason,
        "decisions": decision_dict,
        "artifacts": {},
        "next_input": decision.input_needed,
        "reason": None,
    }

    return {
        "current_step": "orchestrator_agent",
        "next_action": decision.decision,
        "orchestration_decision": decision_dict,

        # важно: только новый элемент
        "orchestration_history": [decision_dict],

        # важно: только новый лог
        "logs": [log_record],
    }

In [31]:
new_state = orchestrator_agent_node(test_state)
new_state

NameError: name 'test_state' is not defined

In [32]:
def route_by_orchestrator_decision(state: AgentState) -> str:
    next_action = state.get("next_action")

    schema = state.get("schema", {})
    data_summary = state.get("data_summary", {})
    prep_summary = state.get("prep_summary", {})
    text_summary = state.get("text_summary", {})
    modeling_summary = state.get("modeling_summary", {})
    artifacts = state.get("artifacts", {})

    data_done = (
        "n_rows" in data_summary
        and "n_columns" in data_summary
        and all(k in schema for k in ["numeric", "categorical", "text", "datetime"])
    )

    prep_done = all(
        k in prep_summary
        for k in ["missing_strategy", "encoding", "scaling", "created_features"]
    )

    text_done = (
        schema.get("text", []) == []
        or text_summary.get("used") is True
    )

    merge_done = artifacts.get("prepared_dataset_path") is not None

    modeling_done = (
        modeling_summary.get("best_model_name") is not None
    )

    reporting_done = (
        artifacts.get("final_report_path") is not None
    )

    error_exists = state.get("error") is not None

    # 1. Если LLM не выбрала действие
    if next_action is None:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 2. Retry/fallback разрешены только при реальной ошибке
    if next_action in ["retry_current_step", "use_fallback"] and not error_exists:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 3. Если есть ошибка — можно доверить LLM retry/fallback/fail
    if error_exists:
        if next_action in ["retry_current_step", "use_fallback", "fail"]:
            return next_action
        return "retry_current_step"

    # 4. Нельзя повторять data_description, если он уже выполнен
    if next_action == "run_data_description" and data_done:
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 5. Нельзя повторять tabular_preparation, если он уже выполнен
    if next_action == "run_tabular_preparation" and prep_done:
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 6. Нельзя повторять text_preparation, если он уже выполнен
    if next_action == "run_text_preparation" and text_done:
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 7. Нельзя запускать merge до tabular preparation
    if next_action == "merge_features" and not prep_done:
        if not data_done:
            return "run_data_description"
        return "run_tabular_preparation"

    # 8. Нельзя повторять merge, если prepared_dataset_path уже есть
    if next_action == "merge_features" and merge_done:
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 9. Нельзя запускать modeling до merge_features
    if next_action == "run_modeling" and not merge_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        return "merge_features"

    # 10. Нельзя повторять modeling, если модель уже обучена
    if next_action == "run_modeling" and modeling_done:
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 11. Improvement можно запускать только после modeling
    if next_action == "run_improvement" and not modeling_done:
        if not merge_done:
            return "merge_features"
        return "run_modeling"

    # 12. Reporting можно запускать только после modeling
    if next_action == "run_reporting" and not modeling_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        return "run_modeling"

    # 13. Нельзя повторять reporting, если отчет уже создан
    if next_action == "run_reporting" and reporting_done:
        return "finish"

    # 14. Нельзя завершать workflow до создания отчета
    if next_action == "finish" and not reporting_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        return "run_reporting"

    # 15. Если отчет есть, но status еще не success, отправляем в reporting
    if next_action == "finish" and reporting_done and state.get("status") != "success":
        return "run_reporting"

    # 16. Иначе доверяем решению LLM
    return next_action

In [33]:
route_by_orchestrator_decision(new_state)

NameError: name 'new_state' is not defined

In [34]:
def data_description_agent_node(state: AgentState) -> AgentState:
    result = {
        "agent": "data_description_agent",
        "status": "success",
        "skipped": False,
        "summary": "Dataset structure was analyzed.",
        "decisions": {
            "task_type_detected": "classification",
            "text_columns_found": False,
        },
        "artifacts": {
            "data_profile_path": "artifacts/data_profile.json",
        },
        "next_input": {
            "schema": {
                "numeric": ["feature_1", "feature_2"],
                "categorical": ["category_1"],
                "text": [],
                "datetime": [],
            },
            "data_summary": {
                "n_rows": 1000,
                "n_columns": 4,
                "missing": {
                    "feature_1": 0,
                    "feature_2": 12,
                    "category_1": 5,
                    "target": 0,
                },
                "duplicates": 3,
                "outlier_cols": ["feature_2"],
                "target_properties": {
                    "type": "binary",
                    "classes": [0, 1],
                },
            },
        },
        "reason": None,
    }

    return {
        "current_step": "data_description_agent",
        "task_type": "classification",
        "schema": result["next_input"]["schema"],
        "data_summary": result["next_input"]["data_summary"],
        "artifacts": {
            **state.get("artifacts", {}),
            **result["artifacts"],
        },
        "logs": [result],
    }

In [35]:
def tabular_preparation_agent_node(state: AgentState) -> AgentState:
    result = {
        "agent": "tabular_preparation_agent",
        "status": "success",
        "skipped": False,
        "summary": "Numeric and categorical features were prepared.",
        "decisions": {
            "missing_strategy": "median_for_numeric_mode_for_categorical",
            "encoding": "one_hot_encoding",
            "scaling": "standard_scaler",
            "created_features": ["feature_1_x_feature_2", "feature_2_is_missing"],
        },
        "artifacts": {
            "tabular_dataset_path": "artifacts/tabular_prepared.csv",
        },
        "next_input": {
            "prep_summary": {
                "duplicates_action": "dropped",
                "missing_strategy": {
                    "numeric": "median",
                    "categorical": "mode",
                },
                "outlier_strategy": "winsorization",
                "encoding": "one_hot_encoding",
                "scaling": "standard_scaler",
                "target_transform": None,
                "created_features": [
                    "feature_1_x_feature_2",
                    "feature_2_is_missing",
                ],
            },
        },
        "reason": None,
    }

    return {
        "current_step": "tabular_preparation_agent",
        "prep_summary": result["next_input"]["prep_summary"],
        "artifacts": {
            **state.get("artifacts", {}),
            **result["artifacts"],
        },
        "logs": [result],
    }

In [36]:
def text_preparation_agent_node(state: AgentState) -> AgentState:
    result = {
        "agent": "text_preparation_agent",
        "status": "success",
        "skipped": False,
        "summary": "Text features were cleaned and vectorized.",
        "decisions": {
            "vectorization": "tfidf",
            "max_features": 100,
        },
        "artifacts": {
            "text_features_path": "artifacts/text_features.csv",
        },
        "next_input": {
            "text_summary": {
                "used": True,
                "text_columns": state.get("schema", {}).get("text", []),
                "vectorization": "tfidf",
                "text_feature_count": 100,
            },
        },
        "reason": None,
    }

    return {
        "current_step": "text_preparation_agent",
        "text_summary": result["next_input"]["text_summary"],
        "artifacts": {
            **state.get("artifacts", {}),
            **result["artifacts"],
        },
        "logs": [result],
    }

In [37]:
def merge_features_node(state: AgentState) -> AgentState:
    result = {
        "agent": "merge_features",
        "status": "success",
        "skipped": False,
        "summary": "Prepared tabular and text features were merged.",
        "decisions": {
            "used_text_features": state.get("text_summary", {}).get("used", False),
        },
        "artifacts": {
            "prepared_dataset_path": "artifacts/prepared_dataset.csv",
        },
        "next_input": {},
        "reason": None,
    }

    return {
        "current_step": "merge_features",
        "artifacts": {
            **state.get("artifacts", {}),
            **result["artifacts"],
        },
        "logs": [result],
    }

In [38]:
def modeling_agent_node(state: AgentState) -> AgentState:
    result = {
        "agent": "modeling_agent",
        "status": "success",
        "skipped": False,
        "summary": "Several ML models were trained and compared.",
        "decisions": {
            "models_tested": [
                "LogisticRegression",
                "RandomForestClassifier",
                "GradientBoostingClassifier",
            ],
            "best_model": "RandomForestClassifier",
            "primary_metric": "f1",
        },
        "artifacts": {
            "best_model_path": "artifacts/best_model.pkl",
            "predictions_path": "artifacts/predictions.csv",
            "metrics_path": "artifacts/metrics.json",
        },
        "next_input": {
            "modeling_summary": {
                "models_tested": [
                    "LogisticRegression",
                    "RandomForestClassifier",
                    "GradientBoostingClassifier",
                ],
                "primary_metric": "f1",
                "best_model_name": "RandomForestClassifier",
                "validation_metrics": {
                    "f1": 0.83,
                    "roc_auc": 0.88,
                },
                "test_metrics": {
                    "f1": 0.81,
                    "roc_auc": 0.86,
                },
                "retry_recommended": False,
                "improvement_mode": None,
                "business_interpretation": "The model can identify the positive class with acceptable quality.",
            },
        },
        "reason": None,
    }

    return {
        "current_step": "modeling_agent",
        "modeling_summary": result["next_input"]["modeling_summary"],
        "artifacts": {
            **state.get("artifacts", {}),
            **result["artifacts"],
        },
        "logs": [result],
    }

In [39]:
def improvement_agent_node(state: AgentState) -> AgentState:
    result = {
        "agent": "improvement_agent",
        "status": "success",
        "skipped": False,
        "summary": "Model improvement strategy was applied.",
        "decisions": {
            "improvement_mode": "hyperparameter_tuning",
        },
        "artifacts": {},
        "next_input": {
            "modeling_summary": {
                **state.get("modeling_summary", {}),
                "retry_recommended": False,
                "improvement_mode": "hyperparameter_tuning",
            },
        },
        "reason": None,
    }

    return {
        "current_step": "improvement_agent",
        "retry_count": state.get("retry_count", 0) + 1,
        "modeling_summary": result["next_input"]["modeling_summary"],
        "logs": [result],
    }

In [40]:
def reporting_agent_node(state: AgentState) -> AgentState:
    result = {
        "agent": "reporting_agent",
        "status": "success",
        "skipped": False,
        "summary": "Final report was created.",
        "decisions": {
            "report_sections": [
                "business_task",
                "data_summary",
                "preprocessing",
                "model_comparison",
                "business_interpretation",
            ],
        },
        "artifacts": {
            "final_report_path": "artifacts/final_report.md",
        },
        "next_input": {},
        "reason": None,
    }

    return {
        "current_step": "reporting_agent",
        "status": "success",
        "artifacts": {
            **state.get("artifacts", {}),
            **result["artifacts"],
        },
        "logs": [result],
    }

In [41]:
def retry_handler_node(state: AgentState) -> AgentState:
    log_record = {
        "agent": "retry_handler",
        "status": "success",
        "skipped": False,
        "summary": "Retry was selected by orchestrator.",
        "decisions": {},
        "artifacts": {},
        "next_input": {},
        "reason": None,
    }

    return {
        "current_step": "retry_handler",
        "retry_count": state.get("retry_count", 0) + 1,
        "error": None,
        "logs": [log_record],
    }


def fallback_handler_node(state: AgentState) -> AgentState:
    log_record = {
        "agent": "fallback_handler",
        "status": "failed",
        "skipped": False,
        "summary": "Fallback strategy was applied, but workflow was stopped.",
        "decisions": {},
        "artifacts": {},
        "next_input": {},
        "reason": state.get("error"),
    }

    return {
        "current_step": "fallback_handler",
        "status": "failed",
        "error": "Fallback was used because the workflow could not recover.",
        "logs": [log_record],
    }


def failure_handler_node(state: AgentState) -> AgentState:
    log_record = {
        "agent": "failure_handler",
        "status": "failed",
        "skipped": False,
        "summary": "Workflow failed.",
        "decisions": {},
        "artifacts": {},
        "next_input": {},
        "reason": state.get("error"),
    }

    return {
        "current_step": "failure_handler",
        "status": "failed",
        "logs": [log_record],
    }

In [42]:
def build_orchestrator_graph():
    workflow = StateGraph(AgentState)

    # Init nodes
    workflow.add_node("dataset_finder", dataset_finder_node)
    workflow.add_node("previous_run_finder", previous_run_finder_node)
    workflow.add_node("join_init", join_init_node)

    # Main LLM orchestrator node
    workflow.add_node("orchestrator_agent", orchestrator_agent_node)

    # Specialized agent nodes
    workflow.add_node("data_description_agent", data_description_agent_node)
    workflow.add_node("tabular_preparation_agent", tabular_preparation_agent_node)
    workflow.add_node("text_preparation_agent", text_preparation_agent_node)
    workflow.add_node("merge_features", merge_features_node)
    workflow.add_node("modeling_agent", modeling_agent_node)
    workflow.add_node("improvement_agent", improvement_agent_node)
    workflow.add_node("reporting_agent", reporting_agent_node)

    # Error / recovery nodes
    workflow.add_node("retry_handler", retry_handler_node)
    workflow.add_node("fallback_handler", fallback_handler_node)
    workflow.add_node("failure_handler", failure_handler_node)

    # Parallel initialization
    workflow.add_edge(START, "dataset_finder")
    workflow.add_edge(START, "previous_run_finder")

    # ВАЖНО: join должен ждать обе параллельные ветки
    workflow.add_edge(["dataset_finder", "previous_run_finder"], "join_init")

    workflow.add_edge("join_init", "orchestrator_agent")

    # Conditional routing after LLM orchestrator decision
    workflow.add_conditional_edges(
        "orchestrator_agent",
        route_by_orchestrator_decision,
        {
            "run_data_description": "data_description_agent",
            "run_tabular_preparation": "tabular_preparation_agent",
            "run_text_preparation": "text_preparation_agent",
            "merge_features": "merge_features",
            "run_modeling": "modeling_agent",
            "run_improvement": "improvement_agent",
            "run_reporting": "reporting_agent",
            "retry_current_step": "retry_handler",
            "use_fallback": "fallback_handler",
            "finish": END,
            "fail": "failure_handler",
        },
    )

    # Return control to orchestrator after each specialized node
    workflow.add_edge("data_description_agent", "orchestrator_agent")
    workflow.add_edge("tabular_preparation_agent", "orchestrator_agent")
    workflow.add_edge("text_preparation_agent", "orchestrator_agent")
    workflow.add_edge("merge_features", "orchestrator_agent")
    workflow.add_edge("modeling_agent", "orchestrator_agent")
    workflow.add_edge("improvement_agent", "orchestrator_agent")
    workflow.add_edge("reporting_agent", "orchestrator_agent")

    # Retry returns to orchestrator
    workflow.add_edge("retry_handler", "orchestrator_agent")

    # Fallback and failure end the graph
    workflow.add_edge("fallback_handler", END)
    workflow.add_edge("failure_handler", END)

    return workflow.compile()

In [43]:
initial_state = {
    "run_id": None,
    "dataset_path": None,
    "target_column": "target",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": [],
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],
    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

In [44]:
app = build_orchestrator_graph()

final_state = app.invoke(
    initial_state
)

In [200]:
workflow = StateGraph(AgentState)

In [201]:
workflow.add_node("orchestrator_agent", orchestrator_agent_node)

workflow.add_node("data_description_agent", data_description_agent_node)
workflow.add_node("tabular_preparation_agent", tabular_preparation_agent_node)
workflow.add_node("text_preparation_agent", text_preparation_agent_node)
workflow.add_node("merge_features", merge_features_node)
workflow.add_node("modeling_agent", modeling_agent_node)
workflow.add_node("improvement_agent", improvement_agent_node)
workflow.add_node("reporting_agent", reporting_agent_node)

workflow.add_node("retry_handler", retry_handler_node)
workflow.add_node("fallback_handler", fallback_handler_node)
workflow.add_node("failure_handler", failure_handler_node)

In [202]:
workflow.add_edge(START, "orchestrator_agent")

In [203]:
workflow.add_conditional_edges(
    "orchestrator_agent",
    route_by_orchestrator_decision,
    {
        "run_data_description": "data_description_agent",
        "run_tabular_preparation": "tabular_preparation_agent",
        "run_text_preparation": "text_preparation_agent",
        "merge_features": "merge_features",
        "run_modeling": "modeling_agent",
        "run_improvement": "improvement_agent",
        "run_reporting": "reporting_agent",
        "retry_current_step": "retry_handler",
        "use_fallback": "fallback_handler",
        "finish": END,
        "fail": "failure_handler",
    },
)

In [204]:
workflow.add_edge("data_description_agent", "orchestrator_agent")
workflow.add_edge("tabular_preparation_agent", "orchestrator_agent")
workflow.add_edge("text_preparation_agent", "orchestrator_agent")
workflow.add_edge("merge_features", "orchestrator_agent")
workflow.add_edge("modeling_agent", "orchestrator_agent")
workflow.add_edge("improvement_agent", "orchestrator_agent")
workflow.add_edge("reporting_agent", "orchestrator_agent")
workflow.add_edge("retry_handler", "orchestrator_agent")
workflow.add_edge("fallback_handler", "orchestrator_agent")

workflow.add_edge("failure_handler", END)

In [205]:
app = workflow.compile()

In [ ]:
initial_state = {
    "run_id": None,
    "dataset_path": None,
    "target_column": "target",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": [],
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

In [220]:
final_state = app.invoke(initial_state)

In [45]:
final_state["status"]

'success'

In [46]:
for i, item in enumerate(final_state["orchestration_history"], 1):
    print(i, item["decision"], "-", item["reason"])

1 run_data_description - The data_description_agent has not been completed as data_summary and schema are empty.
2 run_tabular_preparation - data_description_agent is completed because data_summary contains n_rows and n_columns, and schema contains numeric, categorical, text, and datetime. The next incomplete step is tabular_preparation_agent.
3 run_text_preparation - tabular_preparation_agent is completed because prep_summary contains missing_strategy, encoding, scaling, and created_features. text_preparation_agent is the next incomplete step, but since schema['text'] is an empty list, it is considered completed according to the rules. However, the decision order requires checking text_preparation_agent. Since schema['text'] is [], the text_preparation_agent is technically completed. Moving to merge_features.
4 merge_features - data_description_agent, tabular_preparation_agent, and text_preparation_agent are all completed. text_preparation_agent is completed because schema['text'] is 

In [47]:
for i, log in enumerate(final_state.get("logs", []), 1):
    print(i, log["agent"], "-", log["status"], "-", log["summary"])

1 dataset_finder - success - Dataset found: ./rent_predictions/airbnb_train.csv
2 previous_run_finder - success - No previous run was found.
3 join_init - success - Initialization steps completed: dataset search and previous run search are done.
4 orchestrator_agent - success - The data_description_agent has not been completed as data_summary and schema are empty.
5 data_description_agent - success - Dataset structure was analyzed.
6 orchestrator_agent - success - data_description_agent is completed because data_summary contains n_rows and n_columns, and schema contains numeric, categorical, text, and datetime. The next incomplete step is tabular_preparation_agent.
7 tabular_preparation_agent - success - Numeric and categorical features were prepared.
8 orchestrator_agent - success - tabular_preparation_agent is completed because prep_summary contains missing_strategy, encoding, scaling, and created_features. text_preparation_agent is the next incomplete step, but since schema['text'] 

In [382]:
print("status:", final_state.get("status"))
print("error:", final_state.get("error"))
print("artifacts:", final_state.get("artifacts"))
print("modeling_summary:", final_state.get("modeling_summary"))

status: success
error: None
artifacts: {'data_profile_path': 'artifacts/data_profile.json', 'tabular_dataset_path': 'artifacts/tabular_prepared.csv', 'prepared_dataset_path': 'artifacts/prepared_dataset.csv', 'best_model_path': 'artifacts/best_model.pkl', 'predictions_path': 'artifacts/predictions.csv', 'metrics_path': 'artifacts/metrics.json', 'final_report_path': 'artifacts/final_report.md'}
modeling_summary: {'models_tested': ['LogisticRegression', 'RandomForestClassifier', 'GradientBoostingClassifier'], 'primary_metric': 'f1', 'best_model_name': 'RandomForestClassifier', 'validation_metrics': {'f1': 0.83, 'roc_auc': 0.88}, 'test_metrics': {'f1': 0.81, 'roc_auc': 0.86}, 'retry_recommended': False, 'improvement_mode': None, 'business_interpretation': 'The model can identify the positive class with acceptable quality.'}


In [332]:
final_state.get("artifacts")

{'data_profile_path': 'artifacts/data_profile.json',
 'tabular_dataset_path': 'artifacts/tabular_prepared.csv',
 'prepared_dataset_path': 'artifacts/prepared_dataset.csv',
 'best_model_path': 'artifacts/best_model.pkl',
 'predictions_path': 'artifacts/predictions.csv',
 'metrics_path': 'artifacts/metrics.json',
 'final_report_path': 'artifacts/final_report.md'}

In [333]:
print("schema:", final_state.get("schema"))
print("data_summary:", final_state.get("data_summary"))
print("current_step:", final_state.get("current_step"))
print("error:", final_state.get("error"))

schema: {'numeric': ['feature_1', 'feature_2'], 'categorical': ['category_1'], 'text': [], 'datetime': []}
data_summary: {'n_rows': 1000, 'n_columns': 4, 'missing': {'feature_1': 0, 'feature_2': 12, 'category_1': 5, 'target': 0}, 'duplicates': 3, 'outlier_cols': ['feature_2'], 'target_properties': {'type': 'binary', 'classes': [0, 1]}}
current_step: orchestrator_agent
error: None
